# 步距Q2&圖.ipynb

步態畢業專題研究筆記。此檔保留探索步驟，已清除輸出、執行資訊及 Colab 中繼資料。

包含獨立實驗與尚未統一的座標定義，不保證由上而下可執行；使用整合流程請從根目錄 README 開始。舊輸出不作為本次驗證結果。


In [ ]:
import os
import pandas as pd
from google.colab import files
import io
import matplotlib.pyplot as plt


In [ ]:
uploaded = files.upload()

all_data = pd.DataFrame()


In [ ]:
file_q2_step_lengths = {}
total_file_count = 0


In [ ]:
for file_name in uploaded.keys():

    print(f"Reading file: {file_name}")


    df = pd.read_csv(io.BytesIO(uploaded[file_name]))


    print(f"檢查文件內容: {df.head()}")


    if 'kpts' not in df.columns:
        print(f"Skipping file {file_name}, 'kpts' column not found.")
        continue


    kpts_columns = df['kpts'].str.split(',', expand=True)


    for i in range(kpts_columns.shape[1]):
        kpts_columns.rename(columns={i: f'kpt_{i+1}'}, inplace=True)


    df = df.join(kpts_columns, rsuffix='_kpts')


    print(f"檢查分割後的數據: {df.head()}")


    def step_x(x1, x2):
        return abs(x2 - x1)


    try:
        x16 = pd.to_numeric(df['kpt_46'], errors='coerce')
        x17 = pd.to_numeric(df['kpt_49'], errors='coerce')
    except KeyError:
        print(f"Skipping file {file_name}, 'kpt_46' or 'kpt_49' column not found.")
        continue


    valid_indices = x16.notnull() & x17.notnull()
    x16 = x16[valid_indices]
    x17 = x17[valid_indices]


    step_lengths = [step_x(x16.iloc[i], x17.iloc[i]) for i in range(len(x16))]


    q2_step_length = pd.Series(step_lengths).quantile(0.50) if step_lengths else 0


    df['Q2_step_length'] = q2_step_length


    file_key = file_name.replace('.csv', '')


    file_q2_step_lengths[file_key] = q2_step_length


    total_file_count += 1


    all_data = pd.concat([all_data, df], ignore_index=True)


In [ ]:
print("步距 Q2 per file:")
for file_key, q2 in file_q2_step_lengths.items():
    print(f"{file_key}: Q2 步距 = {q2}")


In [ ]:
print(f"總筆數: {total_file_count}")
print(f"總 Q2 平均步距: {sum(file_q2_step_lengths.values()) / total_file_count if total_file_count > 0 else 0}")


In [ ]:
output_csv_filename = "步距Q2數據.csv"
all_data.to_csv(output_csv_filename, index=False)
print(f"步距 Q2 數據已保存到 '{output_csv_filename}'")
files.download(output_csv_filename)


In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(file_q2_step_lengths.keys(), file_q2_step_lengths.values(), color='skyblue')
plt.title('Q2 (Median) Step Length per File')
plt.xlabel('File')
plt.ylabel('Q2 (Median) Step Length')
plt.xticks(rotation=90, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
plt.boxplot(file_q2_step_lengths.values())
plt.title('Boxplot of Q2 Step Lengths')
plt.ylabel('Q2 (Median) Step Length')
plt.xticks([1], ['Files'])
plt.tight_layout()
plt.show()
